# Large-Model Windows

The windows of chunked inference (04), scored by the large encoder (06) instead of the base one (03). Nothing is trained. The window scores are turned into the same per-text features as in 04 and saved for the final stacker. Scoring needs a GPU and the 06 run attached (for its model); everything else loads from `ARTIFACTS_DIR`.

In [ ]:
import sys
from pathlib import Path

# Make the repo importable: the local checkout, or a fresh clone on Kaggle/Colab. Then pin versions that must match.
REPO_URL = "https://github.com/BohdanZasymovych/ai-hw3-ai-text-detection.git"
REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "setup_env.py").exists()), None)
if REPO is None:
    REPO = Path("/kaggle/working/hw3" if Path("/kaggle").exists() else "/content/hw3")
    if REPO.exists():
        !git -C {REPO} pull -q
    else:
        !git clone -q --depth 1 {REPO_URL} {REPO}
sys.path.insert(0, str(REPO))

import setup_env

setup_env.install_pins()

import pickle

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer

from utils import (ARTIFACTS_DIR, DATA_DIR, ENCODER, ENV, LABEL_ORDER, LARGE_ENCODER, SEED, cached, compare_f1,
                   compare_groups, ensure_data, ensure_dirs, find_artifact, get_split, length_band, load_key, macro_f1)
from utils.encoder import chunk_encode, predict_logits
from utils.stacking import oof_predict, window_features

ensure_dirs()
ensure_data()
split = get_split()  # fixed train/val split, made and cached on first use
gpu = f"{torch.cuda.device_count()}x {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "none"
print(f"env={ENV}  gpu={gpu}  data={DATA_DIR}  artifacts={ARTIFACTS_DIR}")

## Data

Load the fixed validation split and the test set, the settings and logits of the large encoder (06) and of the base encoder (03), and the windows made in 04. The windows are reused as they are: the large and the base encoder share one tokenizer, which the check below confirms on a few texts.

In [ ]:
train_full = pd.read_csv(DATA_DIR / "train.csv")
val = train_full.iloc[split["val"]].reset_index(drop=True)
del train_full
test = pd.read_csv(DATA_DIR / "test.csv")
label_order = np.array(LABEL_ORDER)

LARGE_KEY = load_key("encoder_large_val_logits")  # settings of the 06 large encoder run
assert LARGE_KEY is not None, "06 results not found: attach the 06_large_model output or copy its artifacts"
assert load_key("encoder_large") == LARGE_KEY, "the saved large model is not the one that made the 06 logits"
logits = {model: {name: np.load(find_artifact(f"{prefix}_{name}_logits", "npy")) for name in ["val", "test"]}
          for model, prefix in [("large", "encoder_large"), ("base", "encoder")]}

sample = val["text"].sample(20, random_state=SEED).tolist()
same_tokens = AutoTokenizer.from_pretrained(LARGE_ENCODER)(sample)["input_ids"] == AutoTokenizer.from_pretrained(ENCODER)(sample)["input_ids"]
assert same_tokens, "large and base tokenizers differ: the 04 windows cannot be reused"

WINDOW, MAX_CHUNKS = 510, 8  # the same windows as in 04
chunk_key = {"tokenizer": ENCODER, "n_val": len(val), "n_test": len(test), "window": WINDOW, "max_chunks": MAX_CHUNKS, "v": 1}
tokenizer = AutoTokenizer.from_pretrained(ENCODER)
chunks = {name: cached(f"chunk_ids_{name}", lambda: chunk_encode(df["text"], tokenizer, WINDOW, MAX_CHUNKS),
                       key={**chunk_key, "part": name})
          for name, df in [("val", val), ("test", test)]}
with open(find_artifact("large_ids_val"), "rb") as f:
    val["band"] = length_band(pickle.load(f)["n_tokens"])

pd.DataFrame({"texts": [len(val), len(test)], "windows": [len(chunks[n]["ids"]) for n in ["val", "test"]]},
             index=["validation", "test"])

## Inference

Run the saved large model on every window of the validation and test texts. The window logits are cached, so the final notebook needs no GPU for them.

In [ ]:
infer_key = {"encoder": LARGE_KEY, **chunk_key}
model_dir = find_artifact("encoder_large", kind="dir")  # the 06 model; needed only while the logits are not cached
window_logits = {name: cached(f"chunk_large_logits_{name}", lambda: predict_logits(model_dir, chunks[name]["ids"]),
                              key={**infer_key, "part": name}, kind="npy")
                 for name in ["val", "test"]}
base_window_logits = {name: np.load(find_artifact(f"chunk_logits_{name}", "npy")) for name in ["val", "test"]}

## Evaluation

Macro-F1 of single windows by their position, for texts with at least 3 windows (each window scored alone), for the base (03) and the large encoder. In 04 the base encoder misread windows from the middle of a text.

In [ ]:
doc, pos = chunks["val"]["doc"], chunks["val"]["pos"]
k = chunks["val"]["n_windows"][doc]
position = pd.Series(np.select([pos == 0, pos == k - 1], ["first", "last"], "middle"))[k >= 3]
truth = val["label"].to_numpy()[doc][k >= 3]
pd.DataFrame({
    model: {p: macro_f1(truth[position == p], label_order[w["val"].argmax(axis=1)][k >= 3][position == p])
            for p in ["first", "middle", "last"]}
    for model, w in [("base (03)", base_window_logits), ("large (06)", window_logits)]
}).style.background_gradient(cmap="Blues", axis=None).format("{:.3f}")

Stackers on the large encoder (logistic regression, out-of-fold with 5-fold cross-validation inside the validation set, as in 04): its head + tail logits alone, with the window features of the base encoder (as in 04), with its own window features, and with both. The final notebook combines these with TF-IDF.

In [ ]:
features = {name: {model: window_features(w[name], chunks[name]["doc"], chunks[name]["n_windows"])
                   for model, w in [("base", base_window_logits), ("large", window_logits)]}
            for name in ["val", "test"]}

y = val["label"].to_numpy()
inputs = {
    "stacker": [logits["large"]["val"]],
    "+ base windows": [logits["large"]["val"], logits["base"]["val"], features["val"]["base"]],
    "+ large windows": [logits["large"]["val"], features["val"]["large"]],
    "+ both windows": [logits["large"]["val"], logits["base"]["val"], features["val"]["base"], features["val"]["large"]],
}
preds = {"encoder large": label_order[logits["large"]["val"].argmax(axis=1)],
         **{name: oof_predict(np.hstack(X), y, SEED) for name, X in inputs.items()}}
compare_f1(y, preds).style.background_gradient(cmap="Blues", axis=None).format("{:.3f}")

Macro-F1 by text length.

In [ ]:
compare_groups(val, preds, "band").style.background_gradient(cmap="Blues", subset=list(preds), vmin=0.8, vmax=1) \
    .format({**{name: "{:.3f}" for name in preds}, "n": "{:,}"})

## Conclusions

_To be written after the run._